# 03a – Naive Baseline: Persistenz (Lag-96)

**Modell:** Prognose für Zeitpunkt *t* = Messung von *t − 96 Schritte* (= gestern, gleiche Uhrzeit).  
Kein Training nötig – rein regelbasiert.  
Ergebnisse werden nach `results/naive/persistence.json` geschrieben.

## 0 · Setup

In [1]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import pandas as pd

from src.data.preprocessing import load_pv_data
from src.data.splitting import time_series_split
from src.models.naive import persistence_forecast
from src.evaluation.metrics import evaluate, evaluate_by_season

P_NOM  = 13500.0
TARGET = "Solarproduktion"
RESULTS_PATH = Path("../results/naive/persistence.json")

## 1 · Daten laden

In [2]:
df = load_pv_data()
print(f"Datenpunkte : {len(df):,}")
print(f"Zeitraum    : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

Datenpunkte : 113,896
Zeitraum    : 2022-02-07 → 2025-06-16


## 2 · Val / Test Grenzen bestimmen

Persistenz braucht keine Trainingsdaten – der Split wird nur genutzt,
um die Evaluierungszeiträume (Val und Test) konsistent mit allen anderen Modellen zu halten.

In [3]:
_, val, test = time_series_split(df, val_frac=0.15, test_frac=0.15)

print(f"Val  : {val['timestamp'].min().date()} → {val['timestamp'].max().date()}  ({len(val):,} Punkte)")
print(f"Test : {test['timestamp'].min().date()} → {test['timestamp'].max().date()}  ({len(test):,} Punkte)")

Val  : 2024-06-15 → 2024-12-16  (17,084 Punkte)
Test : 2024-12-16 → 2025-06-16  (17,085 Punkte)


## 3 · Persistenz-Forecast berechnen

In [4]:
# shift(96) schaut nur rückwärts → kein Leakage
full_pred = persistence_forecast(df.set_index("timestamp")[TARGET], lag=96)

y_val  = val.set_index("timestamp")[TARGET]
y_test = test.set_index("timestamp")[TARGET]

pred_val  = full_pred.reindex(y_val.index)
pred_test = full_pred.reindex(y_test.index)

## 4 · Evaluation

In [5]:
metrics_val  = evaluate(y_val,  pred_val,  p_nom=P_NOM)
metrics_test = evaluate(y_test, pred_test, p_nom=P_NOM)

pd.DataFrame({"val": metrics_val, "test": metrics_test}).T.round(3)

,rmse,mae,mbe,nrmse,nmae,nmbe,r2,mape_daytime,skill_mae,skill_rmse,ramp_mae
val,312.571,149.629,1.554,0.023,0.011,0.005,0.620,1.627,0.0,0.0,82.151
test,314.284,147.840,-2.195,0.023,0.011,-0.007,0.608,1.664,0.0,0.0,71.810


In [6]:
seasonal_val  = evaluate_by_season(y_val,  pred_val,  p_nom=P_NOM)
seasonal_test = evaluate_by_season(y_test, pred_test, p_nom=P_NOM)

print("── Val (nach Saison) ──")
display(seasonal_val.round(3))
print("── Test (nach Saison) ──")
display(seasonal_test.round(3))

── Val (nach Saison) ──


,rmse,mae,mbe,nrmse,nmae,nmbe,r2,mape_daytime,skill_mae,skill_rmse,ramp_mae
Winter,224.177,79.902,-2.207,0.017,0.006,-0.034,-0.360,3.457,0.0,0.0,20.121
Summer,381.534,211.388,-1.453,0.028,0.016,-0.003,0.587,1.475,0.0,0.0,134.245
Autumn,255.046,108.955,4.758,0.019,0.008,0.021,0.595,1.615,0.0,0.0,48.406


── Test (nach Saison) ──


,rmse,mae,mbe,nrmse,nmae,nmbe,r2,mape_daytime,skill_mae,skill_rmse,ramp_mae
Winter,238.852,93.895,-1.036,0.018,0.007,-0.007,0.422,2.048,0.0,0.0,37.787
Spring,356.455,181.577,-4.397,0.026,0.013,-0.010,0.594,1.386,0.0,0.0,90.312
Summer,362.935,206.192,4.768,0.027,0.015,0.009,0.639,2.074,0.0,0.0,124.177


## 5 · Ergebnisse speichern

In [7]:
results = {
    "model": "persistence",
    "val":            metrics_val,
    "test":           metrics_test,
    "val_by_season":  seasonal_val.to_dict(orient="index"),
    "test_by_season": seasonal_test.to_dict(orient="index"),
}

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"Gespeichert: {RESULTS_PATH}")

Gespeichert: ../results/naive/persistence.json
